<img src="https://github.com/milena-seismo/DSB-Unit-03/blob/main/labs/03%20-%20Supervised-Model-Comparison/assets/ga-logo.png?raw=1" style="float: left; margin: 20px; height: 55px">

# Lab: Supervised Learning Model Comparison
---

Recall the data science process.

1. Define the problem.
2. Gather the data.
3. Explore the data.
4. Model the data.
5. Evaluate the model.
6. Answer the problem.

In this lab, we're going to focus mostly on creating (and then comparing) many regression and classification models. Thus, we'll define the problem and gather the data for you.
Most of the questions requiring a written response can be written in 2-3 sentences.

### Step 1: Define the problem.

You are a data scientist with a financial services company. Specifically, you want to leverage data in order to identify potential customers.

If you are unfamiliar with "401(k)s" or "IRAs," these are two types of retirement accounts. Very broadly speaking:
- You can put money for retirement into both of these accounts.
- The money in these accounts gets invested and hopefully has a lot more money in it when you retire.
- These are a little different from regular bank accounts in that there are certain tax benefits to these accounts. Also, employers frequently match money that you put into a 401k.
- If you want to learn more about them, check out [this site](https://www.nerdwallet.com/article/ira-vs-401k-retirement-accounts).

We will tackle one regression problem and one classification problem today.
- Regression: What features best predict one's income?
- Classification: Predict whether or not one is eligible for a 401k.

Check out the data dictionary [here](http://fmwww.bc.edu/ec-p/data/wooldridge2k/401KSUBS.DES).

### NOTE: When predicting `inc`, you should pretend as though you do not have access to the `e401k`, the `p401k` variable, and the `pira` variable. When predicting `e401k`, you may use the entire dataframe if you wish.

### Step 2: Gather the data.

##### 1. Read in the data from the repository.

In [2]:
import pandas as pd
sub_401 = pd.read_csv('https://raw.githubusercontent.com/milena-seismo/DSB-Unit-03/refs/heads/main/labs/03%20-%20Supervised-Model-Comparison/data/401ksubs.csv')


In [3]:
sub_401.head(20)

,e401k,inc,marr,male,age,fsize,nettfa,p401k,pira,incsq,agesq
0,0,13.170,0,0,40,1,4.575,0,1,173.4489,1600
1,1,61.230,0,1,35,1,154.000,1,0,3749.1130,1225
2,0,12.858,1,0,44,2,0.000,0,0,165.3282,1936
3,0,98.880,1,1,44,2,21.800,0,0,9777.2540,1936
4,0,22.614,0,0,53,1,18.450,0,0,511.3930,2809
5,0,15.000,1,0,60,3,0.000,0,0,225.0000,3600
6,0,37.155,1,0,49,5,3.483,0,1,1380.4940,2401
7,0,31.896,1,0,38,5,-2.100,0,0,1017.3550,1444
8,0,47.295,1,0,52,2,5.290,0,1,2236.8170,2704
9,1,29.100,0,1,45,1,29.600,0,1,846.8100,2025


In [11]:
sub_401.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9275 entries, 0 to 9274
Data columns (total 11 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   e401k   9275 non-null   int64  
 1   inc     9275 non-null   float64
 2   marr    9275 non-null   int64  
 3   male    9275 non-null   int64  
 4   age     9275 non-null   int64  
 5   fsize   9275 non-null   int64  
 6   nettfa  9275 non-null   float64
 7   p401k   9275 non-null   int64  
 8   pira    9275 non-null   int64  
 9   incsq   9275 non-null   float64
 10  agesq   9275 non-null   int64  
dtypes: float64(3), int64(8)
memory usage: 797.2 KB


In [12]:
sub_401.describe().T

,count,mean,std,min,25%,50%,75%,max
e401k,9275.0,0.392129,0.488252,0.0000,0.0000,0.000,1.0000,1.000
inc,9275.0,39.254641,24.090002,10.0080,21.6600,33.288,50.1600,199.041
marr,9275.0,0.628571,0.483213,0.0000,0.0000,1.000,1.0000,1.000
male,9275.0,0.204420,0.403299,0.0000,0.0000,0.000,0.0000,1.000
age,9275.0,41.080216,10.299517,25.0000,33.0000,40.000,48.0000,64.000
fsize,9275.0,2.885067,1.525835,1.0000,2.0000,3.000,4.0000,13.000
nettfa,9275.0,19.071675,63.963838,-502.3020,-0.5000,2.000,18.4495,1536.798
p401k,9275.0,0.276226,0.447154,0.0000,0.0000,0.000,1.0000,1.000
pira,9275.0,0.254340,0.435513,0.0000,0.0000,0.000,1.0000,1.000
incsq,9275.0,2121.192483,3001.469424,100.1601,469.1556,1108.091,2516.0255,39617.320


##### 2. What are 2-3 other variables that, if available, would be helpful to have?
/Education level, occupation, and years of work experience would be useful additional variables. These factors could help predict income and whether a person is eligible for an employer-sponsored 401(k) plan./

##### 3. Suppose a peer recommended putting `race` into your model in order to better predict who to target when advertising IRAs and 401(k)s. Why would this be an unethical decision?

##### Using race to decide who receives advertisements for retirement accounts could lead to discrimination by excluding certain racial groups from financial opportunities. Targeting should instead focus on relevant financial needs and eligibility, rather than race.#

## Step 3: Explore the data.

##### 4. When attempting to predict income, which feature(s) would we reasonably not use? Why?

In [13]:
# incsq, because it is just inc squared, so it gives away the target. We also leave out e401k, p401k and pira, because the lab tells us not to use them and they depend on income.

##### 5. What two variables have already been created for us through feature engineering? Come up with a hypothesis as to why subject-matter experts may have done this.
> This need not be a "statistical hypothesis." Just brainstorm why SMEs might have done this!

In [ ]:
#

##### 6. Looking at the data dictionary, one variable description appears to be an error. What is this error, and what do you think the correct value would be?

In [ ]:
# The data dictionary incorrectly labels `inc` as “inc^2” and `age` as “age^2.” Their descriptions should be “income” and “age in years,” respectively. The squared values are already stored in `incsq` and `agesq`.

## Step 4: Model the data. (Part 1: Regression Problem)

Recall:
- Problem: What features best predict one's income?
- When predicting `inc`, you should pretend as though you do not have access to the `e401k`, the `p401k` variable, and the `pira` variable.

##### 7. List all modeling tactics we've learned that could be used to solve a regression problem (as of Wednesday afternoon of Week 6). For each tactic, identify whether it is or is not appropriate for solving this specific regression problem and explain why or why not.

In [ ]:
# - **Multiple linear regression:** Appropriate for predicting continuous income and interpreting relationships between income and the predictors.
# However, it may miss complex nonlinear patterns.

# K-nearest neighbors regression:** Appropriate because it estimates income from people with similar characteristics. Features should be scaled so that large-valued variables do not dominate distances.
# Decision tree regression:** Appropriate because it captures nonlinear relationships and interactions, but an unrestricted tree can overfit.
# Bagged decision trees:** Appropriate because averaging multiple trees trained on bootstrap samples reduces variance compared with a single tree.
# Random forest regression:** Appropriate because it captures nonlinear patterns and reduces correlation between trees by considering random subsets of features at each split.
# AdaBoost regression:** Appropriate because it combines weak regressors sequentially to improve predictions, although it can be sensitive to outliers.

# We must exclude `e401k`, `p401k`, and `pira` as instructed. We should also exclude `incsq` because it is calculated directly from the target, `inc`, and would cause target leakage.

##### 8. Regardless of your answer to number 7, fit at least one of each of the following models to attempt to solve the regression problem above:
    - a multiple linear regression model
    - a k-nearest neighbors model
    - a decision tree
    - a set of bagged decision trees
    - a random forest
    - an Adaboost model
    
> As always, be sure to do a train/test split! In order to compare modeling techniques, you should use the same train-test split on each. I recommend setting a random seed here.

> You may find it helpful to set up a pipeline to try each modeling technique, but you are not required to do so!

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    BaggingRegressor,
    RandomForestRegressor,
    AdaBoostRegressor
)

# Predictors: exclude forbidden columns and income squared
X_reg = sub_401.drop(
    columns=["inc", "incsq", "e401k", "p401k", "pira"]
)
y_reg = sub_401["inc"]

# Same split for every regression model
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg,
    y_reg,
    test_size=0.25,
    random_state=42
)

reg_models = {
    "Linear Regression": make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        LinearRegression()
    ),

    "KNN": make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        KNeighborsRegressor(n_neighbors=5)
    ),

    "Decision Tree": make_pipeline(
        SimpleImputer(strategy="median"),
        DecisionTreeRegressor(random_state=42)
    ),

    "Bagging": make_pipeline(
        SimpleImputer(strategy="median"),
        BaggingRegressor(
            n_estimators=100,
            random_state=42,
            n_jobs=-1
        )
    ),

    "Random Forest": make_pipeline(
        SimpleImputer(strategy="median"),
        RandomForestRegressor(
            n_estimators=200,
            random_state=42,
            n_jobs=-1
        )
    ),

    "AdaBoost": make_pipeline(
        SimpleImputer(strategy="median"),
        AdaBoostRegressor(
            n_estimators=100,
            random_state=42
        )
    )
}

for name, model in reg_models.items():
    model.fit(X_train_reg, y_train_reg)
    print(f"{name} trained")

Linear Regression trained
KNN trained
Decision Tree trained
Bagging trained
Random Forest trained
AdaBoost trained


##### 9. What is bootstrapping?

##### 10. What is the difference between a decision tree and a set of bagged decision trees? Be specific and precise!

##### 11. What is the difference between a set of bagged decision trees and a random forest? Be specific and precise!

##### 12. Why might a random forest be superior to a set of bagged decision trees?
> Hint: Consider the bias-variance tradeoff.

## Step 5: Evaluate the model. (Part 1: Regression Problem)

##### 13. Using RMSE, evaluate each of the models you fit on both the training and testing data.

In [7]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error

reg_results = []

for name, model in reg_models.items():
    train_pred = model.predict(X_train_reg)
    test_pred = model.predict(X_test_reg)

    reg_results.append({
        "Model": name,
        "Train RMSE": np.sqrt(
            mean_squared_error(y_train_reg, train_pred)
        ),
        "Test RMSE": np.sqrt(
            mean_squared_error(y_test_reg, test_pred)
        )
    })

reg_results_df = (
    pd.DataFrame(reg_results)
    .set_index("Model")
    .sort_values("Test RMSE")
)

reg_results_df.round(3)

,Train RMSE,Test RMSE
Model,,
KNN,16.457,20.278
Random Forest,7.628,20.432
Bagging,7.676,20.495
Linear Regression,20.086,20.981
AdaBoost,24.301,25.299
Decision Tree,2.298,27.451


##### 14. Based on training RMSE and testing RMSE, is there evidence of overfitting in any of your models? Which ones?

In [ ]:
# Yes. The decision tree shows the strongest evidence of overfitting, with a training RMSE of 2.298 and a testing RMSE of 27.451.
# Random forest and bagging also show large training–testing gaps. KNN shows a smaller gap, while linear regression and AdaBoost show little evidence of overfitting.

##### 15. Based on everything we've covered so far, if you had to pick just one model as your final model to use to answer the problem in front of you, which one model would you pick? Defend your choice.

In [ ]:
# I would choose KNN because it achieved the lowest testing RMSE (20.278) and showed a smaller training–testing gap than random forest and bagging.
# However, its advantage over those models is small, so I would use cross-validation to confirm the choice.

##### 16. Suppose you wanted to improve the performance of your final model. Brainstorm 2-3 things that, if you had more time, you would attempt.

In [ ]:
# I would tune the number of neighbors, distance metric, and neighbor weighting using cross-validation.
# I would also collect relevant predictors, such as education and work experience, and investigate whether redundant features such as age and agesq affect KNN’s performance.

## Step 4: Model the data. (Part 2: Classification Problem)

Recall:
- Problem: Predict whether or not one is eligible for a 401k.
- When predicting `e401k`, you may use the entire dataframe if you wish.

##### 17. While you're allowed to use every variable in your dataframe, mention at least one disadvantage of using `p401k` in your model.

In [8]:
# Using p401k risks target leakage because participation in a 401(k) plan implies eligibility. Participation information may also be unavailable for potential customers when predictions are needed.

##### 18. List all modeling tactics we've learned that could be used to solve a classification problem (as of Wednesday afternoon of Week 6). For each tactic, identify whether it is or is not appropriate for solving this specific classification problem and explain why or why not.

In [11]:
# Logistic regression: Appropriate because the target is binary.
# It provides an interpretable baseline, but may miss complex nonlinear relationships.
# K-nearest neighbors: Appropriate because eligibility may be similar among people with similar characteristics.
# It requires feature scaling and a suitable choice of neighbors.

SyntaxError: invalid syntax (3498049409.py, line 2)

##### 19. Regardless of your answer to number 18, fit at least one of each of the following models to attempt to solve the classification problem above:
    - a logistic regression model
    - a k-nearest neighbors model
    - a decision tree
    - a set of bagged decision trees
    - a random forest
    - an Adaboost model
    
> As always, be sure to do a train/test split! In order to compare modeling techniques, you should use the same train-test split on each. I recommend using a random seed here.

> You may find it helpful to set up a pipeline to try each modeling technique, but you are not required to do so!

In [10]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    BaggingClassifier,
    RandomForestClassifier,
    AdaBoostClassifier
)

# Predictors and target
X = sub_401.drop(columns=["e401k", "p401k"])
y = sub_401["e401k"]

# One shared split for all six models
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

models = {
    "Logistic Regression": make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        LogisticRegression(max_iter=2000, random_state=42)
    ),

    "KNN": make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        KNeighborsClassifier(n_neighbors=5)
    ),

    "Decision Tree": make_pipeline(
        SimpleImputer(strategy="median"),
        DecisionTreeClassifier(random_state=42)
    ),

    "Bagging": make_pipeline(
        SimpleImputer(strategy="median"),
        BaggingClassifier(
            n_estimators=100,
            random_state=42,
            n_jobs=-1
        )
    ),

    "Random Forest": make_pipeline(
        SimpleImputer(strategy="median"),
        RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            n_jobs=-1
        )
    ),

    "AdaBoost": make_pipeline(
        SimpleImputer(strategy="median"),
        AdaBoostClassifier(
            n_estimators=100,
            random_state=42
        )
    )
}

for name, model in models.items():
    model.fit(X_train, y_train)
    print(f"{name} trained")

Logistic Regression trained
KNN trained
Decision Tree trained
Bagging trained
Random Forest trained
AdaBoost trained


## Step 5: Evaluate the model. (Part 2: Classfication Problem)

##### 20. Suppose our "positive" class is that someone is eligible for a 401(k). What are our false positives? What are our false negatives?

In [ ]:
# A false positive occurs when the model predicts that someone is eligible for a 401(k), but they are actually ineligible. A false negative occurs when the model predicts that someone is ineligible, but they are actually eligible.

##### 21. In this specific case, would we rather minimize false positives or minimize false negatives? Defend your choice.

In [ ]:
# Assuming the goal is to reach as many eligible potential customers as possible, we would prioritize minimizing false negatives. Missing eligible people means losing potential customers, while false positives mainly result in unnecessary advertising costs.

##### 22. Suppose we wanted to optimize for the answer you provided in problem 21. Which metric would we optimize in this case?

In [ ]:
# We should maximize recall for the eligible class because higher recall means fewer eligible people are missed.

##### 23. Suppose that instead of optimizing for the metric in problem 21, we wanted to balance our false positives and false negatives using `f1-score`. Why might [f1-score](https://en.wikipedia.org/wiki/F1_score) be an appropriate metric to use here?

In [4]:
# F1-score is appropriate because it combines precision and recall, rewarding models that identify eligible customers while limiting incorrect positive predictions. It is useful when both missed customers and wasted advertising matter, especially if the classes are imbalanced.

##### 24. Using f1-score, evaluate each of the models you fit on both the training and testing data.

In [5]:
from sklearn.metrics import f1_score

results = []

for name, model in models.items():
    train_predictions = model.predict(X_train)
    test_predictions = model.predict(X_test)

    train_f1 = f1_score(
        y_train, train_predictions, zero_division=0
    )
    test_f1 = f1_score(
        y_test, test_predictions, zero_division=0
    )

    results.append({
        "Model": name,
        "Train F1": train_f1,
        "Test F1": test_f1,
        "F1 Gap": train_f1 - test_f1
    })

results_df = pd.DataFrame(results).set_index("Model")

results_df = results_df.sort_values(
    "Test F1", ascending=False
)

print(results_df.round(3))

                     Train F1  Test F1  F1 Gap
Model                                         
AdaBoost                0.585    0.580   0.005
Bagging                 1.000    0.531   0.469
Random Forest           1.000    0.526   0.474
KNN                     0.661    0.489   0.173
Logistic Regression     0.468    0.484  -0.016
Decision Tree           1.000    0.482   0.518


##### 25. Based on training f1-score and testing f1-score, is there evidence of overfitting in any of your models? Which ones?

In [ ]:
# Yes. The decision tree, bagging, and random forest show strong evidence of overfitting: their training F1-scores are 1.000, while their testing scores are only 0.482–0.531. KNN also shows evidence of overfitting, with training F1 of 0.661 and testing F1 of 0.489. AdaBoost and logistic regression show little evidence of overfitting because their training and testing scores are similar.

##### 26. Based on everything we've covered so far, if you had to pick just one model as your final model to use to answer the problem in front of you, which one model would you pick? Defend your choice.

In [ ]:
# I would choose AdaBoost among the models tested because it achieved the highest testing F1-score (0.580) and had a small training–testing gap (0.005). These results suggest that it generalizes better than the other models in this comparison, although I would confirm the choice using cross-validation.

##### 27. Suppose you wanted to improve the performance of your final model. Brainstorm 2-3 things that, if you had more time, you would attempt.

## Step 6: Answer the problem.

##### BONUS: Briefly summarize your answers to the regression and classification problems. Be sure to include any limitations or hesitations in your answer.

- Regression: What features best predict one's income?
- Classification: Predict whether or not one is eligible for a 401k.